<h1 align="center">Lab 2:  Sexism Identification in Twitter</h1>
<h2 align="center">Session 5. Large Language Models: Prompting and In-context Learning</h2>
<h3 style="display:block; margin-top:5px;" align="center">Natural Language and Information Retrieval</h3>
<h3 style="display:block; margin-top:5px;" align="center">Degree in Data Science</h3>
<h3 style="display:block; margin-top:5px;" align="center">2024-2025</h3>    
<h3 style="display:block; margin-top:5px;" align="center">ETSInf. Universitat Politècnica de València</h3>
<br>

### Put your names here

- Bartosz Stoklosa
- Baranyi Sándor

# DO THE WORK !!

In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
import torch
import pandas as pd
from sklearn.metrics import f1_score


In [ ]:
login(token='')
model_path = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_path, token='')
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map='auto')


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
def perform_incontext_classification(model, tokenizer, prompt, ntokens=8, nseq=1, temp=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=ntokens, num_return_sequences=nseq, temperature=temp)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response

def create_incontext_zero_prompt(task_description, query, context=None, role=None, output=None):
    prompt = ""
    if role: prompt += f"You are an expert {role}.\n\n"
    prompt += f"Task: {task_description}\n\n"
    if context: prompt += f"Context: {context}\n\n"
    if output: prompt += f"Output Format: {output} \n\n"
    prompt += f"Input: {query}\nOutput:"
    return prompt

def create_incontext_few_prompt(task_description, exemplars, query, context=None, role=None, output=None):
    prompt = ""
    if role: prompt += f"You are an expert {role}.\n\n"
    prompt += f"Task: {task_description}\n\n"
    if context: prompt += f"Context: {context}\n\n"
    prompt += "Examples in Input and Output format:\n\n"
    for ex in exemplars:
        prompt += f"Input: {ex['input']}\nOutput: {ex['output']}\n\n"
    if output: prompt += f"Output Format: {output} \n\n"
    prompt += f"Input: {query}\nOutput:"
    return prompt

def output_postprocessing_incontext_zero_s2(output):
    outputp = output.rsplit("Output:", 1)[-1].strip()
    for line in outputp.split('\n'):
        line = line.strip().upper()
        if line.startswith("DIRECT"): return "DIRECT"
        if line.startswith("REPORTED"): return "REPORTED"
        if line.startswith("JUDGEMENTAL"): return "JUDGEMENTAL"
    return "UNK"

def sampling_few_instances(trainData, nexamples=3):
    df = pd.DataFrame({'id': trainData[0], 'tweet': trainData[1], 'label': trainData[2]})
    sample = df.groupby('label').sample(n=1, random_state=42)
    return [{'input': row['tweet'], 'output': row['label']} for _, row in sample.iterrows()]


In [4]:
def incontext_zero_pipeline_task2(model, tokenizer, devData, testData):
    role = "in social psychology and linguistics"
    task = """Sexism Source Intention in tweets is a three-class classification task..."""
    output = "The output is a single label: REPORTED/JUDGEMENTAL/DIRECT."

    def run_pipeline(data, filename):
        ids, texts, labels = data[0], data[1], data[2]
        preds = []
        for i, text in enumerate(texts):
            prompt = create_incontext_zero_prompt(task, text, role=role, output=output)
            response = perform_incontext_classification(model, tokenizer, prompt)
            pred = output_postprocessing_incontext_zero_s2(response)
            preds.append(pred if pred in ["DIRECT", "REPORTED", "JUDGEMENTAL"] else "DIRECT")
            print(f"ID: {ids[i]} | GT: {labels[i] if labels else '?'} | Pred: {pred}")
        pd.DataFrame({'id': ids, 'label': preds, 'tweet': texts, 'test_case': ['EXIST2024'] * len(ids)}).to_csv(filename, index=False)
        return preds

    preds_dev = run_pipeline(devData, 'sexism_dev_predictions_task2.csv')
    score = f1_score(devData[2], preds_dev, average='macro')
    print(f"\nMacro F1-score (dev, zero-shot): {score:.4f}")

    run_pipeline(testData, 'sexism_predictions_task2.csv')


In [5]:
def incontext_few_pipeline_task2(model, tokenizer, trainData, devData, testData):
    role = "in social psychology and linguistics"
    task = """Sexism Source Intention in tweets is a three-class classification task..."""
    output = "The output is a single label: REPORTED/JUDGEMENTAL/DIRECT."

    exemplars = sampling_few_instances(trainData, nexamples=3)

    def run_pipeline(data, filename):
        ids, texts, labels = data[0], data[1], data[2]
        preds = []
        for i, text in enumerate(texts):
            prompt = create_incontext_few_prompt(task, exemplars, text, role=role, output=output)
            response = perform_incontext_classification(model, tokenizer, prompt)
            pred = output_postprocessing_incontext_zero_s2(response)
            preds.append(pred if pred in ["DIRECT", "REPORTED", "JUDGEMENTAL"] else "DIRECT")
            print(f"ID: {ids[i]} | GT: {labels[i] if labels else '?'} | Pred: {pred}")
        pd.DataFrame({'id': ids, 'label': preds, 'tweet': texts, 'test_case': ['EXIST2024'] * len(ids)}).to_csv(filename, index=False)
        return preds

    preds_dev = run_pipeline(devData, 'sexism_dev_predictions_task2_few.csv')
    score = f1_score(devData[2], preds_dev, average='macro')
    print(f"\nMacro F1-score (dev, few-shot): {score:.4f}")

    run_pipeline(testData, 'sexism_predictions_task2_few.csv')


In [12]:
import os
from readerEXIST2025 import EXISTReader
base_path = ""
dataset_path = os.path.join(base_path, "corpora/EXIST_2025_Dataset_V0.3/")

file_train = os.path.join(dataset_path, "EXIST2025_training.json")
file_dev = os.path.join(dataset_path, "EXIST2025_dev.json")
file_test = os.path.join(dataset_path, "EXIST2025_test_clean.json")


reader_train = EXISTReader(file_train)
reader_dev = EXISTReader(file_dev)
reader_test = EXISTReader(file_test)

EnTrainTask2, EnDevTask2, EnTestTask2 = reader_train.get(lang="EN", subtask="2"), reader_dev.get(lang="EN", subtask="2"), reader_test.get(lang="EN", subtask="2")

# Run zero-shot
incontext_zero_pipeline_task2(model, tokenizer, EnDevTask2, EnTestTask2)

# Run few-shot
incontext_few_pipeline_task2(model, tokenizer, EnTrainTask2, EnDevTask2, EnTestTask2)


ValueError: All arrays must be of the same length